# Metaclass Parameters — Advanced Problems with Solutions

This notebook is a **tutorial-style problem set** on passing named parameters in a class header and processing them through metaclasses.

The goal is not just to show working code.

Instead, each larger problem is broken into small logical steps:

1. identify what Python is doing,
2. make a small experiment,
3. observe the result,
4. introduce one complication,
5. solve it,
6. extract a reusable rule.

The examples deliberately include both successful and failing class definitions, because metaclass errors often make more sense when we see exactly where a class-creation keyword ends up.


We will focus on class declarations that look like this:

```python
class Service(
    Base,
    metaclass=SomeMeta,
    option1=...,
    option2=...,
):
    ...
```

The extra named arguments are **class-creation keyword arguments**.

They are not constructor arguments for instances.

They are involved while Python is creating the class object itself.


A useful mental model is that class creation has several possible hooks:

```text
class statement
    |
    v
metaclass.__prepare__
    |
    v
execute class body
    |
    v
metaclass.__new__
    |
    v
base.__init_subclass__
    |
    v
metaclass.__init__
```

The exact details matter, especially when more than one hook wants to consume class-header parameters.


In [1]:
print("Ready for the metaclass-parameter tutorial.")

Ready for the metaclass-parameter tutorial.


---

# Problem 1 — Trace a Parameter Through the Class-Creation Pipeline

We will start with observation rather than abstraction.

Suppose we write:

```python
class Example(metaclass=TracingMeta, mode="strict"):
    pass
```

Which hooks can see `mode="strict"`?


## Step 1 — Make every metaclass hook print what it receives

We will instrument:

- `__prepare__`
- `__new__`
- `__init__`

For now, we will **consume** the keyword instead of forwarding it to `type`.


In [2]:
class TracingMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        print("__prepare__", name, kwargs)
        return {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        print("__new__   ", name, kwargs)
        cls = super().__new__(mcls, name, bases, namespace)
        cls.creation_options = dict(kwargs)
        return cls

    def __init__(cls, name, bases, namespace, **kwargs):
        print("__init__  ", name, kwargs)
        super().__init__(name, bases, namespace)


In [3]:
class PipelineExample(
    metaclass=TracingMeta,
    mode="strict",
    audit=True,
):
    answer = 42


__prepare__ PipelineExample {'mode': 'strict', 'audit': True}
__new__    PipelineExample {'mode': 'strict', 'audit': True}
__init__   PipelineExample {'mode': 'strict', 'audit': True}


The same class-header keywords are visible to multiple metaclass hooks.

This immediately gives us a practical rule:

> If you override metaclass hooks, their signatures should be compatible with the class-creation keywords your API allows.


In [4]:
assert PipelineExample.creation_options == {
    "mode": "strict",
    "audit": True,
}

print(PipelineExample.creation_options)


{'mode': 'strict', 'audit': True}


## Step 2 — Why didn't we forward the keywords?

Try forwarding an arbitrary keyword all the way to `type.__new__` when no base class is prepared to consume it.


In [5]:
class ForwardEverythingMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        return super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )


In [6]:
try:
    class BrokenForwarding(
        metaclass=ForwardEverythingMeta,
        mystery=123,
    ):
        pass
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: BrokenForwarding.__init_subclass__() takes no keyword arguments


The keyword did not magically become a class attribute.

It remained part of the class-creation protocol.

If nobody downstream recognizes it, class creation fails.


---

# Problem 2 — Design a Clean Required-Parameter API

We want a metaclass for protocol classes.

Every concrete protocol class must declare:

```python
protocol="..."
```

For example:

```python
class HttpProtocol(metaclass=ProtocolMeta, protocol="http"):
    ...
```

The value must be a non-empty lowercase identifier.


## Step 1 — Start with the smallest working version

Instead of accepting arbitrary `**kwargs`, make the required parameter explicit.


In [7]:
class ProtocolMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        protocol,
        **kwargs,
    ):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        cls.protocol = protocol
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        protocol,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


In [8]:
class HttpProtocol(
    metaclass=ProtocolMeta,
    protocol="http",
):
    pass


print(HttpProtocol.protocol)


http


## Step 2 — Observe what happens when the required option is missing

The explicit keyword-only parameter gives us a useful Python error automatically.


In [9]:
try:
    class MissingProtocol(metaclass=ProtocolMeta):
        pass
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: ProtocolMeta.__new__() missing 1 required keyword-only argument: 'protocol'


## Step 3 — Add validation

We want values such as:

- `"http"` — valid
- `"web_socket"` — valid
- `"HTTP"` — invalid
- `""` — invalid
- `"not-valid"` — invalid


In [10]:
class ValidatedProtocolMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        protocol,
        **kwargs,
    ):
        if not isinstance(protocol, str):
            raise TypeError("protocol must be a string")

        if not protocol:
            raise ValueError("protocol must not be empty")

        if not protocol.isidentifier():
            raise ValueError(
                "protocol must be a valid identifier"
            )

        if protocol != protocol.lower():
            raise ValueError(
                "protocol must be lowercase"
            )

        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )
        cls.protocol = protocol
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        protocol,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


In [11]:
class WebSocketProtocol(
    metaclass=ValidatedProtocolMeta,
    protocol="web_socket",
):
    pass


assert WebSocketProtocol.protocol == "web_socket"
print(WebSocketProtocol.protocol)


web_socket


In [12]:
for bad_protocol in ["", "HTTP", "not-valid", 123]:
    try:
        class BadProtocol(
            metaclass=ValidatedProtocolMeta,
            protocol=bad_protocol,
        ):
            pass
    except (TypeError, ValueError) as exc:
        print(
            repr(bad_protocol),
            "->",
            type(exc).__name__,
            exc,
        )


'' -> ValueError protocol must not be empty
'HTTP' -> ValueError protocol must be lowercase
'not-valid' -> ValueError protocol must be a valid identifier
123 -> TypeError protocol must be a string


The important idea is not the particular validation rule.

The important idea is **when** the validation happens.

Invalid declarations fail while Python is defining the class, which is usually much earlier and clearer than letting bad configuration survive until an instance is created.


---

# Problem 3 — Let the Metaclass and `__init_subclass__` Share the Header

Now we want two independent configuration layers.

The metaclass owns:

```python
trace=True
```

A base class owns:

```python
category="worker"
```

We want this declaration to work:

```python
class Worker(
    Component,
    trace=True,
    category="worker",
):
    ...
```


## Step 1 — Build the base class first

`Component.__init_subclass__` will consume `category`.


In [13]:
class Component:
    def __init_subclass__(
        cls,
        *,
        category=None,
        **kwargs,
    ):
        super().__init_subclass__(**kwargs)
        cls.category = category


## Step 2 — Build a metaclass that consumes only `trace`

This is the key part.

The metaclass should **not** swallow `category`.

It should remove its own option and forward the remainder.


In [14]:
class TraceMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        trace=False,
        **kwargs,
    ):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )
        cls.trace_enabled = bool(trace)
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        trace=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


## Step 3 — Combine the two ideas

There is one subtle point.

For the metaclass to be used by subclasses of `Component`, `Component` itself should also be created by that metaclass.


In [15]:
class Component(metaclass=TraceMeta):
    def __init_subclass__(
        cls,
        *,
        category=None,
        **kwargs,
    ):
        super().__init_subclass__(**kwargs)
        cls.category = category


In [16]:
class BackgroundWorker(
    Component,
    trace=True,
    category="worker",
):
    pass


print("trace:", BackgroundWorker.trace_enabled)
print("category:", BackgroundWorker.category)

assert BackgroundWorker.trace_enabled is True
assert BackgroundWorker.category == "worker"


trace: True
category: worker


This is one of the most reusable metaclass patterns:

```python
def __new__(..., *, my_option=..., **kwargs):
    # consume my_option
    return super().__new__(..., **kwargs)
```

The current layer owns one part of the class header and cooperates with the layers after it.


---

# Problem 4 — Use `__prepare__` Because the Decision Must Happen Before the Body Runs

Most metaclass parameters can be handled in `__new__`.

But some options affect the namespace used **while the class body is executing**.

For those cases, `__prepare__` is the right hook.

We will create:

```python
class Example(
    metaclass=DuplicateControlMeta,
    allow_redefinition=False,
):
    ...
```

When `allow_redefinition=False`, assigning the same class-body name twice should raise an error.


## Step 1 — Create a custom namespace

The namespace will behave like a dictionary, except that it rejects duplicate keys.


In [17]:
class NoRedefinitionDict(dict):
    def __setitem__(self, key, value):
        if key in self:
            raise NameError(
                f"class-body name {key!r} was defined twice"
            )
        super().__setitem__(key, value)


There is one catch.

Python itself writes special entries such as `__module__` and `__qualname__`.

Those are not usually duplicates, but we can make the namespace more robust by allowing a small set of interpreter-level names to be updated.


In [18]:
class NoRedefinitionDict(dict):
    _special = {
        "__module__",
        "__qualname__",
        "__annotations__",
        "__classcell__",
    }

    def __setitem__(self, key, value):
        if key in self and key not in self._special:
            raise NameError(
                f"class-body name {key!r} was defined twice"
            )
        super().__setitem__(key, value)


## Step 2 — Choose the namespace in `__prepare__`


In [19]:
class DuplicateControlMeta(type):
    @classmethod
    def __prepare__(
        mcls,
        name,
        bases,
        *,
        allow_redefinition=True,
        **kwargs,
    ):
        if allow_redefinition:
            return {}
        return NoRedefinitionDict()

    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        allow_redefinition=True,
        **kwargs,
    ):
        cls = super().__new__(
            mcls,
            name,
            bases,
            dict(namespace),
            **kwargs,
        )
        cls.allow_redefinition = allow_redefinition
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        allow_redefinition=True,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


In [20]:
class NormalRedefinition(
    metaclass=DuplicateControlMeta,
    allow_redefinition=True,
):
    value = 10
    value = 20


assert NormalRedefinition.value == 20
print(NormalRedefinition.value)


20


In [21]:
try:
    class StrictRedefinition(
        metaclass=DuplicateControlMeta,
        allow_redefinition=False,
    ):
        value = 10
        value = 20
except NameError as exc:
    print(type(exc).__name__ + ":", exc)


NameError: class-body name 'value' was defined twice


This problem shows the main reason `__prepare__` exists:

> It can change the mapping in which Python executes the class body.


---

# Problem 5 — Inject Descriptors Correctly

Suppose a metaclass parameter describes fields:

```python
class Product(
    metaclass=FieldMeta,
    fields=("name", "price"),
):
    ...
```

We want the metaclass to install descriptors for those fields.

This problem is interesting because descriptors have the `__set_name__` hook.


## Step 1 — Build a descriptor that remembers its assigned name


In [22]:
class Field:
    def __init__(self):
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value


## Step 2 — First try the easy but subtly different approach

Create the class first, then attach descriptors with `setattr`.


In [23]:
class LateFieldMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        fields=(),
        **kwargs,
    ):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        for field_name in fields:
            setattr(cls, field_name, Field())

        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        fields=(),
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


In [24]:
class LateProduct(
    metaclass=LateFieldMeta,
    fields=("name", "price"),
):
    pass


print("LateProduct.name.name =", LateProduct.name.name)


LateProduct.name.name = None


The descriptor was attached, but its `__set_name__` hook was not automatically triggered.

That automatic hook normally happens during class creation.


## Step 3 — Inject the descriptors before calling `type.__new__`


In [25]:
class FieldMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        fields=(),
        **kwargs,
    ):
        fields = tuple(fields)

        for field_name in fields:
            if not isinstance(field_name, str):
                raise TypeError("field names must be strings")

            if not field_name.isidentifier():
                raise ValueError(
                    f"invalid field name: {field_name!r}"
                )

            if field_name in namespace:
                raise ValueError(
                    f"{name}: {field_name!r} already exists"
                )

            namespace[field_name] = Field()

        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )
        cls.fields = fields
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        fields=(),
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


In [26]:
class Product(
    metaclass=FieldMeta,
    fields=("name", "price"),
):
    pass


print(Product.name.name)
print(Product.price.name)

assert Product.name.name == "name"
assert Product.price.name == "price"


name
price


## Step 4 — Confirm that the descriptors actually work


In [27]:
product = Product()
product.name = "Keyboard"
product.price = 99.5

print(product.name)
print(product.price)

assert product.name == "Keyboard"
assert product.price == 99.5


Keyboard
99.5


The practical lesson is:

> If something depends on normal class-creation hooks such as `__set_name__`, place it in the namespace before calling `type.__new__`.


---

# Problem 6 — Preserve `__classcell__`

Advanced metaclasses sometimes rebuild the namespace.

For example, perhaps we want to remove private helper attributes before the class is created.

That sounds harmless, but there is a special compiler-generated entry named `__classcell__`.

Methods that use zero-argument `super()` may depend on it.


## Step 1 — Build a filtering metaclass

We will preserve important interpreter keys explicitly.


In [28]:
class PublicOnlyMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        public_only=False,
        **kwargs,
    ):
        if not public_only:
            cleaned = dict(namespace)
        else:
            cleaned = {}

            always_keep = {
                "__module__",
                "__qualname__",
                "__classcell__",
            }

            for key, value in namespace.items():
                if key in always_keep:
                    cleaned[key] = value
                elif not key.startswith("_"):
                    cleaned[key] = value

        return super().__new__(
            mcls,
            name,
            bases,
            cleaned,
            **kwargs,
        )

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        public_only=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


## Step 2 — Test with zero-argument `super()`

That is exactly the kind of method that makes preserving `__classcell__` important.


In [29]:
class Parent:
    def describe(self):
        return "parent"


class Child(
    Parent,
    metaclass=PublicOnlyMeta,
    public_only=True,
):
    _temporary = "remove me"

    def describe(self):
        return super().describe() + " -> child"


child = Child()

print(child.describe())
print(hasattr(Child, "_temporary"))

assert child.describe() == "parent -> child"
assert not hasattr(Child, "_temporary")


parent -> child
False


When possible, mutating the original namespace is simpler.

If you intentionally build a replacement namespace, treat compiler-generated entries carefully.


---

# Problem 7 — Build an Inheritance-Aware Configuration Parameter

Now we will make a parameter that can be inherited.

The class header supports:

```python
timeout=...
```

Rules:

- root classes default to `30`
- subclasses inherit the nearest base timeout
- a subclass can override it
- explicitly passing `None` is invalid


## Step 1 — Why `None` is not enough as the default

If we wrote:

```python
timeout=None
```

we could not distinguish:

- the user omitted `timeout`
- the user explicitly wrote `timeout=None`

Those are different cases in this API.


We will therefore use a unique sentinel object.


In [30]:
MISSING = object()

## Step 2 — Resolve the effective value


In [31]:
class TimeoutMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        timeout=MISSING,
        **kwargs,
    ):
        if timeout is MISSING:
            inherited_timeout = next(
                (
                    base.timeout
                    for base in bases
                    if hasattr(base, "timeout")
                ),
                30,
            )
            resolved_timeout = inherited_timeout
        else:
            if (
                not isinstance(timeout, (int, float))
                or isinstance(timeout, bool)
                or timeout <= 0
            ):
                raise ValueError(
                    "timeout must be a positive number"
                )

            resolved_timeout = timeout

        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )
        cls.timeout = resolved_timeout
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        timeout=MISSING,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


In [32]:
class RootClient(metaclass=TimeoutMeta):
    pass


class SlowClient(RootClient, timeout=120):
    pass


class SlowChild(SlowClient):
    pass


print(RootClient.timeout)
print(SlowClient.timeout)
print(SlowChild.timeout)

assert RootClient.timeout == 30
assert SlowClient.timeout == 120
assert SlowChild.timeout == 120


30
120
120


## Step 3 — Confirm that explicit `None` is different from omission


In [33]:
try:
    class InvalidTimeout(RootClient, timeout=None):
        pass
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)


ValueError: timeout must be a positive number


The sentinel pattern is useful far beyond metaclasses.

Whenever an API needs to distinguish *missing* from *explicitly passed `None`*, a unique object is usually the cleanest solution.


---

# Problem 8 — Build a Registry Without Accidental Collisions

A common metaclass use case is registering classes.

We want:

```python
class JsonRenderer(
    Renderer,
    renderer_name="json",
):
    ...
```

The registry should map names to classes.


## Step 1 — Start with a base class that should not be registered

We will use an `abstract=True` class-header option.


In [34]:
from types import MappingProxyType

In [35]:
class RendererMeta(type):
    _registry = {}

    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        renderer_name=None,
        abstract=False,
        **kwargs,
    ):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        cls.renderer_name = renderer_name
        cls.abstract = bool(abstract)

        if not abstract:
            if not isinstance(renderer_name, str):
                raise TypeError(
                    "concrete renderers require renderer_name"
                )

            if not renderer_name:
                raise ValueError(
                    "renderer_name must not be empty"
                )

            if renderer_name in mcls._registry:
                previous = mcls._registry[renderer_name]
                raise ValueError(
                    f"{renderer_name!r} is already registered "
                    f"by {previous.__name__}"
                )

            mcls._registry[renderer_name] = cls

        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        renderer_name=None,
        abstract=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)

    @classmethod
    def registry(mcls):
        return MappingProxyType(dict(mcls._registry))


In [36]:
class Renderer(
    metaclass=RendererMeta,
    abstract=True,
):
    pass


class JsonRenderer(
    Renderer,
    renderer_name="json",
):
    pass


class XmlRenderer(
    Renderer,
    renderer_name="xml",
):
    pass


print(RendererMeta.registry())


{'json': <class '__main__.JsonRenderer'>, 'xml': <class '__main__.XmlRenderer'>}


## Step 2 — Test the collision rule


In [37]:
try:
    class AnotherJsonRenderer(
        Renderer,
        renderer_name="json",
    ):
        pass
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)


ValueError: 'json' is already registered by JsonRenderer


## Step 3 — Why expose a read-only registry view?

Returning the internal dictionary directly would let callers mutate metaclass state accidentally.

A mapping proxy gives callers a read-only view.


In [38]:
registry = RendererMeta.registry()

print(registry["json"].__name__)
print(registry["xml"].__name__)

try:
    registry["yaml"] = object
except TypeError as exc:
    print("Read-only registry:", exc)


JsonRenderer
XmlRenderer
Read-only registry: 'mappingproxy' object does not support item assignment


---

# Problem 9 — Compose Two Parameterized Metaclasses

We will now make two reusable metaclass behaviors:

- `LabelMeta` consumes `label=...`
- `SecurityMeta` consumes `secure=...`

Then we will combine them using multiple inheritance.


The challenge is that both metaclasses need to participate.

That means neither one should jump directly to `type.__new__`.

They should use cooperative `super()`.


In [39]:
class LabelMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        label=None,
        **kwargs,
    ):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )
        cls.label = label
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        label=None,
        **kwargs,
    ):
        super().__init__(
            name,
            bases,
            namespace,
            **kwargs,
        )


In [40]:
class SecurityMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        secure=False,
        **kwargs,
    ):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )
        cls.secure = bool(secure)
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        secure=False,
        **kwargs,
    ):
        super().__init__(
            name,
            bases,
            namespace,
            **kwargs,
        )


## Step 2 — Create the combined metaclass

Its method resolution order will decide how the cooperative calls flow.


In [41]:
class CombinedMeta(LabelMeta, SecurityMeta):
    pass


print(
    [item.__name__ for item in CombinedMeta.__mro__]
)


['CombinedMeta', 'LabelMeta', 'SecurityMeta', 'type', 'object']


In [42]:
class Endpoint(
    metaclass=CombinedMeta,
    label="payments",
    secure=True,
):
    pass


print(Endpoint.label)
print(Endpoint.secure)

assert Endpoint.label == "payments"
assert Endpoint.secure is True


payments
True


Each metaclass consumed one keyword and forwarded the rest.

That is exactly what makes the two behaviors composable.


---

# Problem 10 — Build a Class API That Generates Validated Properties

We will create a more realistic metaclass.

The class declaration looks like this:

```python
class Sensor(
    metaclass=TypedFieldsMeta,
    fields={
        "name": str,
        "reading": float,
    },
):
    ...
```

The metaclass will generate descriptors that enforce types.


## Step 1 — Build a typed descriptor


In [43]:
class TypedField:
    def __init__(self, expected_type):
        self.expected_type = expected_type
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self

        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name} must be "
                f"{self.expected_type.__name__}"
            )

        instance.__dict__[self.name] = value


## Step 2 — Validate the field specification

The configuration itself should fail early if it is malformed.


In [44]:
class TypedFieldsMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        fields=None,
        **kwargs,
    ):
        fields = {} if fields is None else dict(fields)

        for field_name, expected_type in fields.items():
            if not isinstance(field_name, str):
                raise TypeError(
                    "field names must be strings"
                )

            if not field_name.isidentifier():
                raise ValueError(
                    f"invalid field name: {field_name!r}"
                )

            if not isinstance(expected_type, type):
                raise TypeError(
                    f"type for {field_name!r} "
                    "must be a class"
                )

            if field_name in namespace:
                raise ValueError(
                    f"{field_name!r} conflicts "
                    "with the class body"
                )

            namespace[field_name] = TypedField(
                expected_type
            )

        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        cls.field_types = MappingProxyType(fields)
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        fields=None,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


## Step 3 — Use the generated fields


In [45]:
class Sensor(
    metaclass=TypedFieldsMeta,
    fields={
        "name": str,
        "reading": float,
    },
):
    pass


sensor = Sensor()
sensor.name = "outdoor"
sensor.reading = 18.5

print(sensor.name, sensor.reading)


outdoor 18.5


## Step 4 — Trigger a runtime field-validation error

The **schema** was validated at class-creation time.

Actual field values are validated when assigned to an instance.


In [46]:
try:
    sensor.reading = "hot"
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: reading must be float


---

# Problem 11 — Use a Metaclass Parameter to Control Instance Creation

Most of this notebook has modified class construction.

A metaclass can also control what happens when the resulting class is called.

Remember:

```python
SomeClass(...)
```

is implemented through the metaclass's `__call__`.


We will create a parameter:

```python
cache_instances=True
```

When enabled, repeated construction with the same positional and keyword arguments should return the same object.


## Step 1 — Store the policy on each class


In [47]:
class InstanceCacheMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        cache_instances=False,
        **kwargs,
    ):
        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        cls.cache_instances = bool(cache_instances)
        cls._instance_cache = {}
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        cache_instances=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)

    def __call__(cls, *args, **kwargs):
        if not cls.cache_instances:
            return super().__call__(*args, **kwargs)

        key = (
            args,
            tuple(sorted(kwargs.items())),
        )

        if key not in cls._instance_cache:
            cls._instance_cache[key] = super().__call__(
                *args,
                **kwargs,
            )

        return cls._instance_cache[key]


## Step 2 — Create a cache-enabled class


In [48]:
class Color(
    metaclass=InstanceCacheMeta,
    cache_instances=True,
):
    def __init__(self, r, g, b):
        self.rgb = (r, g, b)


red1 = Color(255, 0, 0)
red2 = Color(255, 0, 0)
blue = Color(0, 0, 255)

print(red1 is red2)
print(red1 is blue)

assert red1 is red2
assert red1 is not blue


True
False


This particular implementation assumes the constructor arguments are hashable.

That limitation is useful to state explicitly in a production API.


---

# Problem 12 — Enforce a Class-Level Contract

We want a metaclass parameter:

```python
requires=("serialize", "deserialize")
```

The metaclass should verify that the completed class provides callable attributes with those names.

Inherited implementations count.


## Step 1 — Create the class first

We want inherited attributes to count, so checking only the raw namespace would be insufficient.

Instead:

1. create the class,
2. inspect the final class,
3. reject it if the contract is not satisfied.


In [49]:
class ContractMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        requires=(),
        **kwargs,
    ):
        requires = tuple(requires)

        if not all(
            isinstance(item, str)
            for item in requires
        ):
            raise TypeError(
                "requires must contain strings"
            )

        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        problems = []

        for required_name in requires:
            if not hasattr(cls, required_name):
                problems.append(
                    f"{required_name!r} is missing"
                )
            elif not callable(
                getattr(cls, required_name)
            ):
                problems.append(
                    f"{required_name!r} is not callable"
                )

        if problems:
            raise TypeError(
                f"{name}: " + "; ".join(problems)
            )

        cls.requires = requires
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        requires=(),
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


## Step 2 — Demonstrate inherited requirements


In [50]:
class SerializerBase:
    def serialize(self):
        return "serialized"


class Serializer(
    SerializerBase,
    metaclass=ContractMeta,
    requires=("serialize", "deserialize"),
):
    def deserialize(self):
        return "deserialized"


obj = Serializer()

print(obj.serialize())
print(obj.deserialize())


serialized
deserialized


## Step 3 — Demonstrate a contract failure


In [51]:
try:
    class BrokenSerializer(
        metaclass=ContractMeta,
        requires=("serialize", "deserialize"),
    ):
        serialize = 123
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: BrokenSerializer: 'serialize' is not callable; 'deserialize' is missing


---

# Problem 13 — Dynamically Create a Class with Metaclass Parameters

Class-header syntax is not the only way to create classes.

The standard library provides `types.new_class`.

We will use it to create a class dynamically while still passing a custom metaclass parameter.


In [52]:
from types import new_class

We will reuse `TimeoutMeta`.

The dynamically created class should have:

- `timeout=75`
- a method `ping()`


In [53]:
def build_dynamic_namespace(namespace):
    def ping(self):
        return f"pong after {type(self).timeout}s"

    namespace["ping"] = ping


DynamicClient = new_class(
    "DynamicClient",
    (),
    {
        "metaclass": TimeoutMeta,
        "timeout": 75,
    },
    build_dynamic_namespace,
)


In [54]:
dynamic_client = DynamicClient()

print(DynamicClient.timeout)
print(dynamic_client.ping())

assert DynamicClient.timeout == 75
assert dynamic_client.ping() == "pong after 75s"


75
pong after 75s


The important point is that class-creation parameters are part of Python's class-construction machinery, not merely a special syntax trick attached to the `class` statement.


---

# Problem 14 — Decide Whether a Metaclass Is Actually Necessary

Suppose we only want this:

```python
class UserModel(BaseModel, table="users"):
    ...
```

Do we need a metaclass?

Often, no.

`__init_subclass__` is a simpler tool when the goal is only to configure subclasses.


## Step 1 — Solve it without a metaclass


In [55]:
class BaseModel:
    table = None

    def __init_subclass__(
        cls,
        *,
        table=None,
        **kwargs,
    ):
        super().__init_subclass__(**kwargs)

        if table is not None:
            if not isinstance(table, str):
                raise TypeError(
                    "table must be a string"
                )

            if not table:
                raise ValueError(
                    "table must not be empty"
                )

        cls.table = table


In [56]:
class UserModel(
    BaseModel,
    table="users",
):
    pass


print(UserModel.table)
assert UserModel.table == "users"


users


This solution is smaller and easier to compose than introducing a custom metaclass.

A useful heuristic is:

- use `__init_subclass__` for ordinary subclass configuration,
- use class decorators for post-creation transformation,
- use metaclasses when you need deeper control over class creation itself.


---

# Problem 15 — Advanced Debugging: Find Who Owns Each Keyword

Consider this declaration:

```python
class Task(
    BaseTask,
    metaclass=TaskMeta,
    tracing=True,
    queue="critical",
    retries=3,
):
    ...
```

Suppose class creation fails with an unexpected-keyword error.

How do we debug it systematically?


A good debugging strategy is to assign ownership.

For example:

```text
tracing  -> metaclass
queue    -> BaseTask.__init_subclass__
retries  -> another base hook
```

Every keyword should have a clear destination.


We can build a small demonstrator that logs what each layer consumes.


In [57]:
class RetryMixin:
    def __init_subclass__(
        cls,
        *,
        retries=0,
        **kwargs,
    ):
        print(
            "RetryMixin consumed retries =",
            retries,
        )
        super().__init_subclass__(**kwargs)
        cls.retries = retries


class QueueBase:
    def __init_subclass__(
        cls,
        *,
        queue="default",
        **kwargs,
    ):
        print(
            "QueueBase consumed queue =",
            queue,
        )
        super().__init_subclass__(**kwargs)
        cls.queue = queue


In [58]:
class TaskMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        tracing=False,
        **kwargs,
    ):
        print(
            "TaskMeta consumed tracing =",
            tracing,
        )

        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        cls.tracing = tracing
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        tracing=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


We need a root base that uses the custom metaclass so the final subclass can inherit from it cleanly.


In [59]:
class TaskRoot(
    RetryMixin,
    QueueBase,
    metaclass=TaskMeta,
):
    pass


TaskMeta consumed tracing = False
RetryMixin consumed retries = 0
QueueBase consumed queue = default


In [60]:
class CriticalTask(
    TaskRoot,
    tracing=True,
    queue="critical",
    retries=3,
):
    pass


print(
    CriticalTask.tracing,
    CriticalTask.queue,
    CriticalTask.retries,
)


TaskMeta consumed tracing = True
RetryMixin consumed retries = 3
QueueBase consumed queue = critical
True critical 3


When debugging metaclass parameters, ask:

1. Which hook first receives the keyword?
2. Which hook owns it?
3. Is that hook consuming it?
4. Which keywords remain?
5. Is the remainder intentionally forwarded?
6. Does every downstream hook accept what reaches it?

That turns a vague metaclass error into a routing problem.


---

# Problem 16 — A Production-Style Mini Framework

We will finish with a larger example that combines several ideas.

We want event classes declared like this:

```python
class UserCreated(
    Event,
    event_name="user.created",
    version=2,
    aliases=("user.new",),
):
    ...
```

Requirements:

- the root `Event` is abstract,
- concrete events require a unique `event_name`,
- aliases must also be unique,
- version must be a positive integer,
- metadata should be read-only,
- events should be resolvable by either their primary name or alias.


## Step 1 — Define the metaclass registry

We will keep one registry mapping every accepted event name to its class.


In [61]:
class EventMeta(type):
    _registry = {}

    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        abstract=False,
        event_name=None,
        version=1,
        aliases=(),
        **kwargs,
    ):
        if (
            not isinstance(version, int)
            or isinstance(version, bool)
            or version <= 0
        ):
            raise ValueError(
                "version must be a positive integer"
            )

        aliases = tuple(aliases)

        if not all(
            isinstance(alias, str) and alias
            for alias in aliases
        ):
            raise TypeError(
                "aliases must contain non-empty strings"
            )

        if event_name is not None:
            if (
                not isinstance(event_name, str)
                or not event_name
            ):
                raise TypeError(
                    "event_name must be a non-empty string"
                )

        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        if not abstract:
            if event_name is None:
                raise TypeError(
                    f"{name} requires event_name"
                )

            names = (event_name, *aliases)

            if len(set(names)) != len(names):
                raise ValueError(
                    f"{name}: duplicate names "
                    "inside event declaration"
                )

            collisions = [
                item
                for item in names
                if item in mcls._registry
            ]

            if collisions:
                raise ValueError(
                    f"{name}: event names already "
                    f"registered: {collisions}"
                )

            for item in names:
                mcls._registry[item] = cls

        cls.metadata = MappingProxyType(
            {
                "abstract": bool(abstract),
                "event_name": event_name,
                "version": version,
                "aliases": aliases,
            }
        )

        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        abstract=False,
        event_name=None,
        version=1,
        aliases=(),
        **kwargs,
    ):
        super().__init__(name, bases, namespace)

    @classmethod
    def resolve(mcls, event_name):
        return mcls._registry[event_name]


## Step 2 — Create the abstract root

The root can still define ordinary instance behavior.


In [62]:
class Event(
    metaclass=EventMeta,
    abstract=True,
):
    def describe(self):
        return (
            f"{self.metadata['event_name']} "
            f"v{self.metadata['version']}"
        )


## Step 3 — Create concrete event classes


In [63]:
class UserCreated(
    Event,
    event_name="user.created",
    version=2,
    aliases=("user.new",),
):
    pass


class UserDeleted(
    Event,
    event_name="user.deleted",
    version=1,
):
    pass


In [64]:
print(UserCreated().describe())
print(UserDeleted().describe())

assert (
    EventMeta.resolve("user.created")
    is UserCreated
)

assert (
    EventMeta.resolve("user.new")
    is UserCreated
)

assert (
    EventMeta.resolve("user.deleted")
    is UserDeleted
)


user.created v2
user.deleted v1


## Step 4 — Confirm metadata cannot be mutated through the exposed mapping


In [65]:
try:
    UserCreated.metadata["version"] = 99
except TypeError as exc:
    print("Read-only metadata:", exc)


Read-only metadata: 'mappingproxy' object does not support item assignment


## Step 5 — Confirm registry collision protection


In [66]:
try:
    class DuplicateUserEvent(
        Event,
        event_name="user.new",
    ):
        pass
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)


ValueError: DuplicateUserEvent: event names already registered: ['user.new']


---

# Additional Advanced Exercises

The next problems are shorter, but each demonstrates another design pattern.


## Exercise A — Normalize a Parameter Before Storing It

Create a metaclass with:

```python
tags=("API", " Internal ", "api")
```

Normalize the tags by:

- stripping whitespace,
- lowercasing,
- removing duplicates while preserving order.


### Solution


In [67]:
class NormalizedTagsMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        tags=(),
        **kwargs,
    ):
        normalized = []
        seen = set()

        for tag in tags:
            if not isinstance(tag, str):
                raise TypeError(
                    "tags must contain strings"
                )

            cleaned = tag.strip().lower()

            if not cleaned:
                raise ValueError(
                    "tags must not be empty"
                )

            if cleaned not in seen:
                seen.add(cleaned)
                normalized.append(cleaned)

        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        cls.tags = tuple(normalized)
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        tags=(),
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


In [68]:
class TaggedService(
    metaclass=NormalizedTagsMeta,
    tags=("API", " Internal ", "api"),
):
    pass


print(TaggedService.tags)
assert TaggedService.tags == (
    "api",
    "internal",
)


('api', 'internal')


## Exercise B — Cross-Parameter Validation

Build a retry policy with:

- `retries`
- `backoff`
- `jitter`

If `jitter=True`, require both `retries > 0` and `backoff > 0`.


### Solution


In [69]:
class RetryPolicyMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        retries=0,
        backoff=0.0,
        jitter=False,
        **kwargs,
    ):
        if (
            not isinstance(retries, int)
            or isinstance(retries, bool)
            or retries < 0
        ):
            raise ValueError(
                "retries must be a non-negative integer"
            )

        if (
            not isinstance(backoff, (int, float))
            or isinstance(backoff, bool)
            or backoff < 0
        ):
            raise ValueError(
                "backoff must be a non-negative number"
            )

        if not isinstance(jitter, bool):
            raise TypeError(
                "jitter must be bool"
            )

        if jitter and (
            retries == 0
            or backoff == 0
        ):
            raise ValueError(
                "jitter=True requires "
                "retries > 0 and backoff > 0"
            )

        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        cls.retry_policy = MappingProxyType(
            {
                "retries": retries,
                "backoff": float(backoff),
                "jitter": jitter,
            }
        )

        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        retries=0,
        backoff=0.0,
        jitter=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


In [70]:
class ReliableClient(
    metaclass=RetryPolicyMeta,
    retries=4,
    backoff=0.5,
    jitter=True,
):
    pass


print(dict(ReliableClient.retry_policy))


{'retries': 4, 'backoff': 0.5, 'jitter': True}


## Exercise C — Parameter-Controlled Method Injection

If `debug=True`, inject a `debug_state()` instance method.

If the user already defined `debug_state`, do not overwrite it.


### Solution


In [71]:
class DebugMethodMeta(type):
    def __new__(
        mcls,
        name,
        bases,
        namespace,
        *,
        debug=False,
        **kwargs,
    ):
        if debug and "debug_state" not in namespace:
            def debug_state(self):
                return {
                    "class": type(self).__name__,
                    "attributes": dict(self.__dict__),
                }

            namespace["debug_state"] = debug_state

        cls = super().__new__(
            mcls,
            name,
            bases,
            namespace,
            **kwargs,
        )

        cls.debug = bool(debug)
        return cls

    def __init__(
        cls,
        name,
        bases,
        namespace,
        *,
        debug=False,
        **kwargs,
    ):
        super().__init__(name, bases, namespace)


In [72]:
class Debuggable(
    metaclass=DebugMethodMeta,
    debug=True,
):
    pass


d = Debuggable()
d.value = 10

print(d.debug_state())


{'class': 'Debuggable', 'attributes': {'value': 10}}


---

# Best-Practices Checklist

When designing a class API with metaclass parameters, ask these questions.


### 1. Who owns each keyword?

Do not let several layers accidentally interpret the same parameter.

For example:

```text
metaclass:
    trace

base class:
    category

another base:
    retries
```

Clear ownership makes forwarding predictable.


### 2. Can the parameter be explicit instead of hidden in `**kwargs`?

Prefer:

```python
def __new__(..., *, trace=False, **kwargs):
    ...
```

over extracting everything manually from a generic dictionary when the option is part of the public API.


### 3. Does the decision need to happen before the class body executes?

If yes, consider `__prepare__`.

If no, `__new__` is usually simpler.


### 4. Does the change depend on normal class-creation hooks?

Descriptors are a good example.

If you want automatic `__set_name__`, inject them before `type.__new__`.


### 5. Are you rebuilding the namespace?

If so, preserve interpreter-significant entries, especially `__classcell__`.


### 6. Do you actually need a metaclass?

Try simpler mechanisms first:

- `__init_subclass__`
- class decorators
- ordinary base classes
- descriptors

Metaclasses are most valuable when the behavior genuinely belongs to class construction or metaclass-level behavior.


### 7. Are multiple metaclasses expected to cooperate?

Use `super()` consistently.

Hard-coding `type.__new__` inside a reusable intermediate metaclass can bypass another metaclass in the MRO.


### 8. Have you tested failure cases?

Good metaclass APIs fail early for:

- invalid types,
- invalid combinations,
- duplicate registry keys,
- unsupported names,
- missing required parameters,
- accidental namespace conflicts.


---

# Final Mini-Quiz

Consider:

```python
class Report(
    BaseReport,
    metaclass=ReportMeta,
    format="pdf",
    version=3,
    cache=True,
):
    ...
```

Before implementing anything, write down:

```text
format  -> ?
version -> ?
cache   -> ?
```

Then decide:

- which layer consumes each option,
- which options need inheritance,
- which options require class-body-time behavior,
- which options should be forwarded,
- which options should never reach `type` unchanged.


A strong metaclass design usually becomes much easier once parameter ownership is explicit.

That is the central theme of this notebook.


In [73]:
print(
    "Tutorial complete:",
    "advanced metaclass parameters."
)


Tutorial complete: advanced metaclass parameters.
